# Настройка окружения

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q pandas numpy scikit-learn catboost shap statsmodels \
                pyarrow openpyxl plotly joblib scipy

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd
import numpy as np
import re
import joblib
from pathlib import Path
from scipy.optimize import brentq

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from catboost import CatBoostRegressor

In [6]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [7]:
PROJECT = Path('/content/drive/MyDrive/Projects/golden-meadows')
RAW = PROJECT / 'data/raw'
TEMP = PROJECT / 'data/temp'
PROCESSED = PROJECT / 'data/processed'
MODELS = PROJECT / 'models'
REPORTS = PROJECT / 'reports'

# for p in [RAW, TEMP, PROCESSED, MODELS, REPORTS]:
#     p.mkdir(parents=True, exist_ok=True)

# print("Окружение готово")

# ETL — сборка панели

## Функции очистки

In [8]:
def norm_kt(x):
    if pd.isna(x): return x
    s = str(x).strip().lower()
    digits = ''.join(ch for ch in s if ch.isdigit())
    return f'КТ{int(digits)}' if digits else s

In [9]:
def to_num(x):
    """Очистка: 'более 105' -> 105, 'не обнар' -> NaN, '< 2000' -> 2000."""
    if pd.isna(x): return np.nan
    s = str(x).strip().replace(',', '.')
    miss = ['-', '', 'не обнар', 'не обруж.', 'не обнаружены',
            'не обнаружен', 'не обн', 'nan', 'none']
    if s.lower() in miss: return np.nan
    s = re.sub(r'(более|менее|не менее|не более)\s*', '', s, flags=re.I)
    s = re.sub(r'^[<>]\s*', '', s)
    s = re.sub(r'[^0-9.\-]', '', s)
    s = re.sub(r'-+$', '', s)
    try: return float(s)
    except (ValueError, TypeError): return np.nan

In [10]:
def parse_date(x):
    if pd.isna(x): return pd.NaT
    s = str(x).strip()
    if re.fullmatch(r'\d{4}', s):
        return pd.Timestamp(int(s), 7, 1)
    try:
        return pd.to_datetime(s, dayfirst=True, errors='coerce')
    except Exception:
        return pd.NaT

## Сточная вода

In [11]:
wastewater = pd.read_csv(RAW / 'wastewater.csv', encoding='UTF-8')
wastewater.columns = [c.strip() for c in wastewater.columns]

In [12]:
renamed = {
    'к.т': 'КТ', 'к.т.': 'КТ', 'дата': 'Дата', 'Дата': 'Дата',
    'БПК5 мг/дм3': 'БПК5', 'ХПК мгО/дм3': 'ХПК',
    'Водородный показатель (рН) ед.рН': 'pH',
    'Массовая доля сульфатов  мг/дм3 (SO4)': 'Сульфаты',
    'Массовая доля жиров мг/дм3': 'Жиры',
    'Фосфаты мг/дм3 (РО4)': 'Фосфаты',
    'Взвешенные вещества мг/дм3': 'Взвешенные',
    'Аммонийный азот мг/дм3': 'Аммоний',
    'Нитраты мг/дм3': 'Нитраты', 'Хлориды мг/дм3': 'Хлориды',
    'Калий мг/дм3': 'Калий', 'Натрий мг/дм3': 'Натрий',
    'Литий мг/дм3': 'Литий', 'Магний мг/дм3': 'Магний',
    'Кальций мг/дм3': 'Кальций', 'Стронций мг/дм3': 'Стронций',
    'Цинк мг/дм3': 'Цинк', 'Марганец мг/дм3': 'Марганец',
    'Нитриты мг/дм3': 'Нитриты', 'Фторид-ион мг/дм3': 'Фториды',
    'Барий мг/кг': 'Барий',
}

In [13]:
wastewater = wastewater.rename(columns=renamed)
wastewater = wastewater.loc[:, ~wastewater.columns.duplicated()]
wastewater['КТ'] = wastewater['КТ'].map(norm_kt)
wastewater = wastewater[~wastewater['КТ'].isin(['норматив', 'пдк почв мг/кг', '-'])]
wastewater = wastewater.dropna(subset=['КТ', 'Дата'])
wastewater['Дата'] = wastewater['Дата'].map(parse_date)
wastewater = wastewater.dropna(subset=['Дата'])

In [14]:
for c in wastewater.columns:
    if c not in ('КТ', 'Дата'):
        wastewater[c] = wastewater[c].map(to_num)

In [15]:
wastewater['Начало_месяца'] = wastewater['Дата'].dt.to_period('M').dt.to_timestamp()
num_cols_w = [c for c in wastewater.columns if c not in ('КТ', 'Дата', 'Начало_месяца')]
w_month = (wastewater.groupby(['КТ', 'Начало_месяца'])[num_cols_w]
             .mean(numeric_only=True).reset_index())

In [16]:
wastewater_wide = w_month.pivot(index='Начало_месяца', columns='КТ', values=num_cols_w)
wastewater_wide.columns = [f'{col}_{kt}' for col, kt in wastewater_wide.columns]
wastewater_wide = wastewater_wide.reset_index()

In [17]:
print(f"wastewater: {wastewater_wide.shape}, {wastewater_wide['Начало_месяца'].min().date()} — {wastewater_wide['Начало_месяца'].max().date()}")

wastewater: (22, 106), 2022-04-01 — 2025-06-01


In [18]:
wastewater

,КТ,Дата,Сульфаты,Жиры,pH,БПК5,ХПК,Фосфаты,Взвешенные,Аммоний,...,Литий,Магний,Кальций,Стронций,Цинк,Марганец,Нитриты,Фториды,Барий,Начало_месяца
0,КТ1,2022-04-05,357.50,18640.00,7.50,937.0,1586.0,184.0,790.0,23.94,...,51.06,269.2,130.3,NaN,0.32,0.52,NaN,NaN,NaN,2022-04-01
1,КТ2,2022-04-05,68.17,7760.00,9.00,342.0,950.0,115.2,250.0,125.30,...,NaN,86.1,167.5,93.8,0.30,0.56,NaN,NaN,NaN,2022-04-01
2,КТ1,2022-06-06,128.70,50.10,7.31,446.0,1360.0,0.1,900.0,39.40,...,NaN,75.6,128.9,NaN,0.04,0.07,NaN,NaN,NaN,2022-06-01
3,КТ2,2022-06-06,333.30,110.00,7.36,199.0,540.0,2.6,880.0,52.90,...,NaN,86.6,93.1,NaN,0.08,0.07,NaN,NaN,NaN,2022-06-01
4,КТ3,2022-06-06,100.00,40.00,7.34,114.0,275.0,3.4,270.0,63.70,...,NaN,70.3,126.3,NaN,0.11,0.16,NaN,NaN,NaN,2022-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,КТ4,2025-05-11,26.50,0.68,6.50,367.0,1600.0,14.5,4560.0,20.20,...,NaN,30.2,115.0,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-01
138,КТ1,2025-05-11,180.20,0.53,6.60,346.0,510.0,25.3,1800.0,6.20,...,NaN,48.6,135.0,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-01
139,КТ2,2025-05-11,167.50,0.50,6.10,1350.0,1630.0,15.3,5040.0,16.90,...,NaN,25.5,112.5,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-01
140,КТ3,2025-05-11,46.80,0.58,6.30,620.0,1260.0,12.5,3200.0,21.30,...,NaN,29.8,120.4,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-01


## Сточная вода работы дозирующих устройств

In [19]:
wastewater_dosing = pd.read_csv(RAW / 'wastewater-of-dosing-devices.csv',
                encoding='utf-8')
wastewater_dosing.columns = [c.strip() for c in wastewater_dosing.columns]
wastewater_dosing = wastewater_dosing.rename(columns={wastewater_dosing.columns[0]: 'Точка'})
wastewater_dosing['Точка'] = wastewater_dosing['Точка'].astype(str).str.strip().str.lower()
wastewater_dosing['Точка'] = wastewater_dosing['Точка'].str.replace(r'\s+', ' ', regex=True)

In [20]:
for c in wastewater_dosing.columns:
    if c not in ('Точка', 'Дата'):
        wastewater_dosing[c] = wastewater_dosing[c].map(to_num)

In [21]:
wastewater_dosing['Дата'] = wastewater_dosing['Дата'].map(parse_date)
wastewater_dosing = wastewater_dosing.dropna(subset=['Дата'])
wastewater_dosing['Начало_месяца'] = wastewater_dosing['Дата'].dt.to_period('M').dt.to_timestamp()

In [22]:
keep_points = ['1 кнс', '2 кнс', 'слив в карту', 'колодец перед 1 кнс']
wastewater_dosing = wastewater_dosing[wastewater_dosing['Точка'].isin(keep_points)]

In [23]:
num_cols_d = [c for c in wastewater_dosing.columns if c not in ('Точка', 'Дата', 'Начало_месяца')]
d_month = (wastewater_dosing.groupby(['Точка', 'Начало_месяца'])[num_cols_d]
             .mean(numeric_only=True).reset_index())

In [24]:
wastewater_dosing_wide = d_month.pivot(index='Начало_месяца', columns='Точка', values=num_cols_d)
wastewater_dosing_wide.columns = [f'{col}_{pt}' for col, pt in wastewater_dosing_wide.columns]
wastewater_dosing_wide.columns = (wastewater_dosing_wide.columns
                  .str.replace(' ', '_')
                  .str.replace('1_кнс', 'КНС1')
                  .str.replace('2_кнс', 'КНС2')
                  .str.replace('слив_в_карту', 'Слив')
                  .str.replace('колодец_перед_1_кнс', 'Колодец'))
wastewater_dosing_wide = wastewater_dosing_wide.reset_index()

print(f"КНС: {wastewater_dosing_wide.shape}")

КНС: (17, 85)


In [25]:
wastewater_dosing_wide

,Начало_месяца,Массовая_доля_сульфатов__мг/дм3_(SO4)_КНС1,Массовая_доля_сульфатов__мг/дм3_(SO4)_КНС2,Массовая_доля_сульфатов__мг/дм3_(SO4)_колодец_перед_КНС1,Массовая_доля_сульфатов__мг/дм3_(SO4)_Слив,Массовая_доля_жиров_мг/дм3_КНС1,Массовая_доля_жиров_мг/дм3_КНС2,Массовая_доля_жиров_мг/дм3_колодец_перед_КНС1,Массовая_доля_жиров_мг/дм3_Слив,Водородный_показатель_(рН)_ед.рН_КНС1,...,Нитриты_мг/дм3_колодец_перед_КНС1,Нитриты_мг/дм3_Слив,Фторид-ион_мг/дм3_КНС1,Фторид-ион_мг/дм3_КНС2,Фторид-ион_мг/дм3_колодец_перед_КНС1,Фторид-ион_мг/дм3_Слив,Барий_мг/кг_КНС1,Барий_мг/кг_КНС2,Барий_мг/кг_колодец_перед_КНС1,Барий_мг/кг_Слив
0,2023-06-01,297.80,96.03,NaN,336.45,0.328,2.30,NaN,1.24,6.60,...,NaN,46.775,NaN,0.75,NaN,4.72,2.7,2.70,NaN,2.68
1,2023-07-01,441.70,212.40,NaN,1550.00,354.000,9.15,NaN,155.00,8.40,...,NaN,NaN,NaN,0.58,NaN,NaN,NaN,0.20,NaN,NaN
2,2023-08-01,1496.00,185.00,NaN,1522.00,298.000,6.50,NaN,135.00,7.80,...,NaN,1.400,NaN,0.45,NaN,3.20,1.0,1.20,NaN,0.60
3,2024-05-01,260.65,205.20,NaN,199.25,1.305,13.95,NaN,0.26,7.20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09,NaN,NaN
4,2024-06-01,268.90,169.80,NaN,208.90,0.380,0.40,NaN,1.16,8.96,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-07-01,505.50,373.10,NaN,233.50,2.090,0.62,NaN,0.39,9.82,...,NaN,NaN,250.3,88.90,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-08-01,368.90,866.50,NaN,203.50,1.060,1.12,NaN,0.48,6.34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-09-01,268.90,466.50,NaN,198.20,0.560,0.65,NaN,0.42,6.81,...,NaN,NaN,NaN,NaN,NaN,NaN,0.5,NaN,NaN,0.70
8,2024-10-01,353.10,525.60,NaN,209.80,0.980,0.80,NaN,0.68,6.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.20
9,2024-11-01,325.70,653.80,NaN,185.50,1.100,0.95,NaN,0.90,8.20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Сводная таблица стоков

In [26]:
WASTEWATER_SOURCES = ['моющее средство', 'сыворотка', 'творожный цех', 'цельномолочный цех', 'котельная']

In [27]:
inflows = pd.read_csv(RAW / 'inflows-summary.csv',
                encoding='utf-8')

months_dict = {
    'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4, 'май': 5, 'июнь': 6,
    'июль': 7, 'август': 8, 'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
}
inflows['Номер_месяца'] = inflows['Месяц'].map(months_dict)

inflows = inflows.dropna(subset=['Номер_месяца'])
inflows['Номер_месяца'] = inflows['Номер_месяца'].astype(int)
inflows = inflows.sort_values(['Год', 'Номер_месяца']).reset_index(drop=True)
inflows['общий_объём'] = inflows[WASTEWATER_SOURCES].sum(axis=1, min_count=1)
inflows['Дата'] = pd.to_datetime(dict(year=inflows['Год'],
                                    month=inflows['Номер_месяца'], day=1))

In [28]:
inflows = inflows[['Дата'] + WASTEWATER_SOURCES + ['общий_объём']].rename(columns={'Дата': 'Начало_месяца'})
print(f"Объёмы: {inflows.shape}, до {inflows['Начало_месяца'].max().date()}")

Объёмы: (60, 7), до 2026-12-01


In [29]:
inflows.columns = (inflows.columns.str.replace(' ', '_'))

In [30]:
inflows

,Начало_месяца,моющее_средство,сыворотка,творожный_цех,цельномолочный_цех,котельная,общий_объём
0,2024-01-01,30.0,1002.0,7650.0,22250.0,2000.0,32932.0
1,2024-02-01,35.0,956.0,8965.0,24406.0,2000.0,36362.0
2,2024-03-01,43.0,997.0,11200.0,33560.0,2000.0,47800.0
3,2024-04-01,39.0,1134.0,10760.0,35750.0,2000.0,49683.0
4,2024-05-01,42.0,1086.0,12340.0,30153.0,2000.0,45621.0
5,2024-06-01,39.0,1196.0,7338.0,33275.0,2000.0,43848.0
6,2024-07-01,39.0,1137.0,10610.0,36568.0,2000.0,50354.0
7,2024-08-01,42.0,1256.0,10760.0,30269.0,2000.0,44327.0
8,2024-09-01,33.0,1137.0,11390.0,28468.0,2000.0,43028.0
9,2024-10-01,37.0,1265.0,13280.0,30056.0,2000.0,46638.0


## Погода

In [31]:
weather = pd.read_csv(RAW / 'tobolsk-weather-2000-2025.csv', encoding='utf-8-sig')
weather.columns = [c.strip() for c in weather.columns][:4]
weather = weather.rename(columns={weather.columns[0]: 'Дата', weather.columns[1]: 'Температура_воздуха',
                         weather.columns[2]: 'Осадки_мм', weather.columns[3]: 'Снежный_покров_мм'})
weather['Дата'] = weather['Дата'].map(parse_date)
weather = weather.dropna(subset=['Дата'])
for c in ['Температура_воздуха', 'Осадки_мм', 'Снежный_покров_мм']:
    weather[c] = pd.to_numeric(weather[c], errors='coerce')

weather['Начало_месяца'] = weather['Дата'].dt.to_period('M').dt.to_timestamp()
weather_monthly = weather.groupby('Начало_месяца').agg(
    T_mean=('Температура_воздуха', 'mean'),
    T_min=('Температура_воздуха', 'min'),
    T_max=('Температура_воздуха', 'max'),
    T_sum_pos=('Температура_воздуха', lambda s: s[s > 0].sum()),
    T_sum_neg=('Температура_воздуха', lambda s: s[s < 0].sum()),
    precip_sum=('Осадки_мм', 'sum'),
    precip_days=('Осадки_мм', lambda s: (s > 0).sum()),
    snow_max=('Снежный_покров_мм', 'max'),
).reset_index()

print(f"Погода: {weather_monthly.shape}, до {weather_monthly['Начало_месяца'].max().date()}")

Погода: (301, 9), до 2025-03-01


In [32]:
weather

,Дата,Температура_воздуха,Осадки_мм,Снежный_покров_мм,Начало_месяца
0,2000-05-04,4.5,0.4,20.0,2000-05-01
1,2000-06-04,-2.3,0.0,15.0,2000-06-01
2,2000-07-04,-1.3,0.0,11.0,2000-07-01
3,2000-08-04,0.7,0.0,9.0,2000-08-01
4,2000-09-04,-0.6,0.7,9.0,2000-09-01
...,...,...,...,...,...
5566,2025-03-27,4.8,0.0,NaN,2025-03-01
5567,2025-03-28,3.2,0.0,NaN,2025-03-01
5568,2025-03-29,4.1,0.0,NaN,2025-03-01
5569,2025-03-30,5.4,0.0,NaN,2025-03-01


## Донные отложения

In [33]:
bottom_sediments = pd.read_csv(RAW / 'bottom-sediments.csv', encoding='utf-8-sig')
bottom_sediments.columns = [c.strip() for c in bottom_sediments.columns]
bottom_sediments = bottom_sediments.rename(columns={bottom_sediments.columns[0]: 'КТ', bottom_sediments.columns[1]: 'Дата'})
bottom_sediments['КТ'] = bottom_sediments['КТ'].map(norm_kt)
bottom_sediments = bottom_sediments[~bottom_sediments['КТ'].isin(['норматив', 'пдк почв мг/кг',
                         'фон на смеж участке', '-'])]
bottom_sediments['Дата'] = bottom_sediments['Дата'].map(parse_date)
bottom_sediments = bottom_sediments.dropna(subset=['Дата', 'КТ'])
for c in bottom_sediments.columns:
    if c not in ('КТ', 'Дата'):
        bottom_sediments[c] = bottom_sediments[c].map(to_num)

bottom_sediments['Начало_месяца'] = bottom_sediments['Дата'].dt.to_period('M').dt.to_timestamp()
num_cols_bs = [c for c in bottom_sediments.columns if c not in ('КТ', 'Дата', 'Начало_месяца')]
bs_month = (bottom_sediments.groupby(['КТ', 'Начало_месяца'])[num_cols_bs]
              .mean(numeric_only=True).reset_index())

bottom_sediments_wide = bs_month.pivot(index='Начало_месяца', columns='КТ', values=num_cols_bs)
bottom_sediments_wide.columns = [f'SED_{col}_{kt}' for col, kt in bottom_sediments_wide.columns]
bottom_sediments_wide = bottom_sediments_wide.reset_index()

print(f"Донные отложения: {bottom_sediments_wide.shape}")

Донные отложения: (27, 106)


In [34]:
bottom_sediments_wide.columns = (bottom_sediments_wide.columns.str.replace(' ', '_'))

In [35]:
bottom_sediments_wide

,Начало_месяца,SED_Массовая_доля_серы_мг/кг_КТ1,SED_Массовая_доля_серы_мг/кг_КТ2,SED_Массовая_доля_серы_мг/кг_КТ3,SED_Массовая_доля_серы_мг/кг_КТ4,SED_Массовая_доля_серы_мг/кг_КТ5,SED_Водородный_показатель_рН_ед.рН_КТ1,SED_Водородный_показатель_рН_ед.рН_КТ2,SED_Водородный_показатель_рН_ед.рН_КТ3,SED_Водородный_показатель_рН_ед.рН_КТ4,...,SED_Массовая_доля_аммонийного_азота_мг/кг_(обменный)_КТ1,SED_Массовая_доля_аммонийного_азота_мг/кг_(обменный)_КТ2,SED_Массовая_доля_аммонийного_азота_мг/кг_(обменный)_КТ3,SED_Массовая_доля_аммонийного_азота_мг/кг_(обменный)_КТ4,SED_Массовая_доля_аммонийного_азота_мг/кг_(обменный)_КТ5,SED_Зола_%_КТ1,SED_Зола_%_КТ2,SED_Зола_%_КТ3,SED_Зола_%_КТ4,SED_Зола_%_КТ5
0,2019-07-01,NaN,NaN,2000.00,NaN,NaN,NaN,NaN,6.50,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-05-01,NaN,600.000000,700.00,NaN,NaN,NaN,7.800000,7.70,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-07-01,600.000000,633.333333,700.00,NaN,NaN,7.800,7.766667,7.70,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-05-01,NaN,24.000000,NaN,NaN,NaN,NaN,6.800000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-06-01,8843.800000,11345.000000,13098.10,11053.60,NaN,7.200,7.100000,6.40,7.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2022-07-01,2718.000000,5633.000000,12610.00,5613.00,NaN,8.400,7.600000,7.10,7.700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2022-08-01,7251.000000,7315.000000,18009.50,12870.00,NaN,8.500,7.200000,6.40,7.600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2022-09-01,1172.000000,5030.000000,7547.00,415.40,NaN,8.200,7.100000,6.50,7.400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2022-10-01,861.700000,2471.000000,5089.00,7963.00,3297.000,8.200,7.400000,7.50,7.700,...,148.30,448.10,406.10,518.70,773.40,97.70,75.80,69.40,75.10,81.60
9,2022-11-01,2806.000000,3339.000000,15230.00,5847.00,989.200,8.100,7.300000,7.50,7.400,...,119.72,452.89,785.26,694.03,156.21,14.50,78.00,65.40,74.00,56.82


## Расход препаратов

In [36]:
drug_consumption = pd.read_csv(RAW / 'drug-consumption.csv')
months_dict = {
    'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4, 'май': 5, 'июнь': 6,
    'июль': 7, 'август': 8, 'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
}
drug_consumption['Номер_месяца'] = drug_consumption['Месяц'].map(months_dict)
drug_consumption.head(10)

,Год,Месяц,Препарат,Количество,Номер_месяца
0,2024,май,Аквамицин А,35.0,5
1,2024,май,Аквамицин С,17.5,5
2,2024,июнь,Аквамицин А,45.0,6
3,2024,июнь,Аквамицин С,22.5,6
4,2024,июль,Аквамицин А,25.0,7
5,2024,июль,Аквамицин С,12.5,7
6,2024,август,Аквамицин А,72.5,8
7,2024,август,Аквамицин С,40.0,8
8,2024,сентябрь,Аквамицин А,50.0,9
9,2024,сентябрь,Аквамицин С,25.0,9


In [37]:
drug_consumption['флаг_экос'] = drug_consumption['Препарат'].str.contains('Экос', case=False, na=False)

In [38]:
drug_consumption

,Год,Месяц,Препарат,Количество,Номер_месяца,флаг_экос
0,2024,май,Аквамицин А,35.0,5,False
1,2024,май,Аквамицин С,17.5,5,False
2,2024,июнь,Аквамицин А,45.0,6,False
3,2024,июнь,Аквамицин С,22.5,6,False
4,2024,июль,Аквамицин А,25.0,7,False
5,2024,июль,Аквамицин С,12.5,7,False
6,2024,август,Аквамицин А,72.5,8,False
7,2024,август,Аквамицин С,40.0,8,False
8,2024,сентябрь,Аквамицин А,50.0,9,False
9,2024,сентябрь,Аквамицин С,25.0,9,False


In [39]:
drug_consumption_monthly = (drug_consumption.groupby(['Год', 'Номер_месяца'])
                  .agg(
                      total_l=('Количество', 'sum'),
                      aquamicin_l=('Количество', lambda s:
                          s[~drug_consumption.loc[s.index, 'флаг_экос']].sum()),
                      ecos_l=('Количество', lambda s:
                          s[drug_consumption.loc[s.index, 'флаг_экос']].sum()),
                  )
                  .reset_index())

drug_consumption_monthly['Начало_месяца'] = pd.to_datetime(dict(
    year=drug_consumption_monthly['Год'],
    month=drug_consumption_monthly['Номер_месяца'],
    day=1))

In [40]:
drug_consumption_monthly

,Год,Номер_месяца,total_l,aquamicin_l,ecos_l,Начало_месяца
0,2024,5,52.5,52.5,0.0,2024-05-01
1,2024,6,67.5,67.5,0.0,2024-06-01
2,2024,7,37.5,37.5,0.0,2024-07-01
3,2024,8,112.5,112.5,0.0,2024-08-01
4,2024,9,75.0,75.0,0.0,2024-09-01
5,2024,10,60.0,60.0,0.0,2024-10-01
6,2025,4,37.5,37.5,0.0,2025-04-01
7,2025,5,52.5,52.5,0.0,2025-05-01
8,2025,6,45.0,45.0,0.0,2025-06-01
9,2025,7,85.0,30.0,55.0,2025-07-01


In [41]:
drug_consumption_monthly.drop('Номер_месяца', axis=1, inplace=True)

In [42]:
drug_consumption_monthly.rename({
    'total_l': 'Всего_препарата_л',
    'aquamicin_l': 'Объём_аквамицина_л',
    'ecos_l': 'Объём_экоса_л',
  },
  axis=1,
  inplace=True
)

In [43]:
drug_consumption_monthly

,Год,Всего_препарата_л,Объём_аквамицина_л,Объём_экоса_л,Начало_месяца
0,2024,52.5,52.5,0.0,2024-05-01
1,2024,67.5,67.5,0.0,2024-06-01
2,2024,37.5,37.5,0.0,2024-07-01
3,2024,112.5,112.5,0.0,2024-08-01
4,2024,75.0,75.0,0.0,2024-09-01
5,2024,60.0,60.0,0.0,2024-10-01
6,2025,37.5,37.5,0.0,2025-04-01
7,2025,52.5,52.5,0.0,2025-05-01
8,2025,45.0,45.0,0.0,2025-06-01
9,2025,85.0,30.0,55.0,2025-07-01


## Сборка панели

In [44]:
panel = (wastewater_wide
         .merge(inflows, on='Начало_месяца', how='outer')
         .merge(weather_monthly, on='Начало_месяца', how='outer')
         .merge(wastewater_dosing_wide, on='Начало_месяца', how='left')
         .merge(bottom_sediments_wide, on='Начало_месяца', how='left')
         .merge(drug_consumption_monthly, on='Начало_месяца', how='left'))

panel = panel.sort_values('Начало_месяца').reset_index(drop=True)

panel['Синус_месяца'] = np.sin(2 * np.pi * panel['Начало_месяца'].dt.month / 12)
panel['Косинус_месяца'] = np.cos(2 * np.pi * panel['Начало_месяца'].dt.month / 12)
panel['Зима'] = panel['Начало_месяца'].dt.month.isin([12, 1, 2]).astype(int)

panel.to_csv(PROCESSED / 'panel.csv', index=False)
print(f"Панель: {panel.shape}")
print(f"Период: {panel['Начало_месяца'].min().date()} — {panel['Начало_месяца'].max().date()}")

Панель: (348, 316)
Период: 2000-01-01 — 2026-12-01


In [45]:
panel

,Начало_месяца,Сульфаты_КТ1,Сульфаты_КТ2,Сульфаты_КТ3,Сульфаты_КТ4,Сульфаты_КТ5,Жиры_КТ1,Жиры_КТ2,Жиры_КТ3,Жиры_КТ4,...,SED_Зола_%_КТ3,SED_Зола_%_КТ4,SED_Зола_%_КТ5,Год,Всего_препарата_л,Объём_аквамицина_л,Объём_экоса_л,Синус_месяца,Косинус_месяца,Зима
0,2000-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.000000e-01,8.660254e-01,1
1,2000-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.660254e-01,5.000000e-01,1
2,2000-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,6.123234e-17,0
3,2000-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.660254e-01,-5.000000e-01,0
4,2000-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.000000e-01,-8.660254e-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,2026-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2026.0,82.0,0.0,82.0,-8.660254e-01,-5.000000e-01,0
344,2026-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000e+00,-1.836970e-16,0
345,2026-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-8.660254e-01,5.000000e-01,0
346,2026-11-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.000000e-01,8.660254e-01,0


# Загрузка панели и Feature-engineering

In [46]:
panel = pd.read_csv(PROCESSED / 'panel.csv')
panel = panel.set_index('Начало_месяца').sort_index()
print(f"Панель: {panel.shape}")

Панель: (348, 315)


In [47]:
df = panel.copy()

In [48]:
for lag in [1, 2, 3]:
    df[f'BOD_KT5_lag{lag}'] = df['БПК5_КТ5'].shift(lag)
    df[f'COD_KT5_lag{lag}'] = df['ХПК_КТ5'].shift(lag)

df['BOD_KT5_roll3'] = df['БПК5_КТ5'].shift(1).rolling(3).mean()
df['BOD_KT5_roll6'] = df['БПК5_КТ5'].shift(1).rolling(6).mean()
df['COD_KT5_roll3'] = df['ХПК_КТ5'].shift(1).rolling(3).mean()
df['BOD_KT5_trend'] = df['БПК5_КТ5'] - df['БПК5_КТ5'].shift(1)

In [49]:
for col in ['БПК5', 'ХПК', 'pH', 'Аммоний', 'Взвешенные', 'Фосфаты']:
    for kt in ['КТ1', 'КТ2', 'КТ3']:
        c = f'{col}_{kt}'
        if c in df.columns:
            df[c] = df[c]  # уже есть
    for kt in ['КТ4']:
        c = f'{col}_{kt}'
        if c in df.columns:
            df[f'{c}_lag1'] = df[c].shift(1)

In [50]:
df['removal_1karta'] = (df['БПК5_КТ1'] - df['БПК5_КТ3']) / df['БПК5_КТ1'].replace(0, np.nan)
df['removal_2karta'] = (df['БПК5_КТ3'].shift(1) - df['БПК5_КТ5'].shift(1)) / df['БПК5_КТ3'].shift(1).replace(0, np.nan)
df['BOD_COD_ratio_KT1'] = df['БПК5_КТ1'] / df['ХПК_КТ1'].replace(0, np.nan)

In [51]:
df['V_общий_lag1'] = df['общий_объём'].shift(1)
df['V_общий_roll3'] = df['общий_объём'].shift(1).rolling(3).mean()
df['share_сыворотка'] = df['сыворотка'] / df['общий_объём'].replace(0, np.nan)

In [52]:
df['load_BOD_KT1'] = df['БПК5_КТ1'] * df['общий_объём'] / 1000
df['load_COD_KT1'] = df['ХПК_КТ1'] * df['общий_объём'] / 1000
df['load_BOD_KT5_lag1'] = df['БПК5_КТ5'].shift(1) * df['общий_объём'].shift(1) / 1000
df['cum_load_BOD_KT5'] = df['load_BOD_KT5_lag1'].fillna(0).cumsum()

In [53]:
if 'dosing_l' in df.columns:
    df['dosing_rate'] = df['Всего_препарата_л'] / df['общий_объём'].replace(0, np.nan)
    df['dosing_lag1'] = df['dosing_rate'].shift(1)
    df['dosing_roll3'] = df['dosing_rate'].shift(1).rolling(3).mean()

In [54]:
df['target_BOD'] = df['БПК5_КТ5'].shift(-1)
df['target_COD'] = df['ХПК_КТ5'].shift(-1)

In [55]:
# Для отложений
sed_target_cols = [c for c in df.columns if c.startswith('SED_')]

# Возьмём только КТ5 — там сброс в болото
sed_cols_kt5 = [c for c in sed_target_cols if 'КТ5' in c]
for c in sed_cols_kt5:
    df[f'target_{c}'] = df[c].shift(-1)

In [56]:
print(f"После Feature Engineering: {df.shape}")
print(f"Target BOD NaN: {df['target_BOD'].isna().sum()}")
print(f"Target COD NaN: {df['target_COD'].isna().sum()}")

После Feature Engineering: (348, 364)
Target BOD NaN: 321
Target COD NaN: 320


# Модуль 1. Прогнозирование БПК5 и ХПК

## Обработка пропусков

In [57]:
EXCLUDE = [c for c in df.columns
           if c.startswith('target_') or c == 'БПК5_КТ5' or c == 'ХПК_КТ5']

final = df.dropna(subset=['target_BOD', 'target_COD']).copy()

feature_cols = [c for c in final.columns if c not in EXCLUDE]
# Отбрасываем признаки с >30% пропусков
nan_share = final[feature_cols].isna().mean()
keep = nan_share[nan_share < 0.3].index.tolist()
feature_cols = keep

X = final[feature_cols].copy()
y_bod = final['target_BOD'].copy()
y_cod = final['target_COD'].copy()

# ffill + median
X = X.ffill().fillna(X.median(numeric_only=True))

print(f"Наблюдений: {len(X)}")
print(f"Признаков: {X.shape[1]}")

Наблюдений: 27
Признаков: 74


## Разбиение на обучающую и тестовую выборки

In [58]:
test_size = max(6, int(np.ceil(len(X) * 0.20)))
split = len(X) - test_size

X_dev, X_test = X.iloc[:split], X.iloc[split:]
y_bod_dev, y_bod_test = y_bod.iloc[:split], y_bod.iloc[split:]
y_cod_dev, y_cod_test = y_cod.iloc[:split], y_cod.iloc[split:]

print(f"Train: {len(X_dev)}, Test: {len(X_test)}")
print(f"Test: {X_test.index.min()} — {X_test.index.max()}")

Train: 21, Test: 6
Test: 2025-04-01 — 2025-06-01


## Метрики и naive baseline

In [59]:
def metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

In [60]:
naive_bod = final['БПК5_КТ5'].iloc[split:].values
naive_cod = final['ХПК_КТ5'].iloc[split:].values

In [61]:
assert len(y_bod_test) == len(naive_bod), f"Разная длина: {len(y_bod_test)} и {len(naive_bod)}"

In [62]:
print("Naive BOD:", {k: round(v, 2) for k, v in metrics(y_bod_test, naive_bod).items()})
print("Naive COD:", {k: round(v, 2) for k, v in metrics(y_cod_test, naive_cod).items()})

Naive BOD: {'MAE': 6.33, 'RMSE': np.float64(11.21), 'R2': -1.23}
Naive COD: {'MAE': 36.67, 'RMSE': np.float64(64.22), 'R2': -0.89}


## Подбор моделей

In [63]:
tscv = TimeSeriesSplit(n_splits=3, gap=1)

def tune_model(model, params, X_dev, y_dev, n_iter=10):
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=n_iter, cv=tscv,
        scoring='neg_mean_absolute_error',
        random_state=RANDOM_STATE, n_jobs=-1, refit=True,
    )
    search.fit(X_dev, y_dev)
    return search

### Ridge

In [64]:
ridge_params = {'model__alpha': np.logspace(-3, 3, 20)}
rf_params = {
    'n_estimators': [200, 400],
    'max_depth': [2, 3, 4],
    'min_samples_leaf': [2, 3, 5],
    'max_features': ['sqrt', 0.7],
}

### CatBoost

In [65]:
cb_params = {
    'iterations': [200, 400],
    'learning_rate': [0.01, 0.03, 0.05],
    'depth': [2, 3, 4],
    'l2_leaf_reg': [5, 10, 20],
    'loss_function': ['MAE'],
}

### Обучение

In [66]:
def train_all(X_dev, y_dev):
    results = {}
    ridge = Pipeline([('sc', StandardScaler()), ('model', Ridge())])
    results['Ridge'] = tune_model(ridge, ridge_params, X_dev, y_dev)
    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=300)
    results['RF'] = tune_model(rf, rf_params, X_dev, y_dev)
    cb = CatBoostRegressor(random_seed=RANDOM_STATE, verbose=False,
                            allow_writing_files=False)
    results['CatBoost'] = tune_model(cb, cb_params, X_dev, y_dev)
    return results

In [67]:
print("Обучаем Модуль 1 для БПК5...")
results_bod = train_all(X_dev, y_bod_dev)
for name, s in results_bod.items():
    print(f"  {name}: CV MAE = {-s.best_score_:.2f}")

Обучаем Модуль 1 для БПК5...
  Ridge: CV MAE = 62.55
  RF: CV MAE = 61.29
  CatBoost: CV MAE = 55.31


In [68]:
print("Обучаем Модуль 1 для ХПК...")
results_cod = train_all(X_dev, y_cod_dev)
for name, s in results_cod.items():
    print(f"  {name}: CV MAE = {-s.best_score_:.2f}")

Обучаем Модуль 1 для ХПК...
  Ridge: CV MAE = 117.86
  RF: CV MAE = 115.94
  CatBoost: CV MAE = 104.36


###  Оценка на финальном тесте

In [69]:
def evaluate_all(results, X_test, y_test, naive):
    rows = [{'Model': 'Naive', **metrics(y_test, naive)}]
    for name, s in results.items():
        pred = s.best_estimator_.predict(X_test)
        rows.append({'Model': name, **metrics(y_test, pred)})
    return pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)

In [70]:
res_bod_df = evaluate_all(results_bod, X_test, y_bod_test, naive_bod)
print(res_bod_df.to_string(index=False))

   Model       MAE      RMSE        R2
   Naive  6.333333 11.210114 -1.234074
      RF  6.562530  8.474965 -0.276889
CatBoost  9.382853 11.744911 -1.452319
   Ridge 12.833013 14.897626 -2.945587


In [71]:
res_cod_df = evaluate_all(results_cod, X_test, y_cod_test, naive_cod)
print(res_cod_df.to_string(index=False))

   Model       MAE      RMSE        R2
   Naive 36.666667 64.219026 -0.886967
   Ridge 41.201676 45.197947  0.065296
      RF 45.796545 53.383239 -0.303907
CatBoost 47.855074 56.740774 -0.473083


In [72]:
best_bod_name = res_bod_df.iloc[0]['Model']
best_cod_name = res_cod_df.iloc[0]['Model']

best_bod = (results_bod[best_bod_name].best_estimator_
            if best_bod_name != 'Naive' else None)
best_cod = (results_cod[best_cod_name].best_estimator_
            if best_cod_name != 'Naive' else None)

print(f"Лучшая для БПК5: {best_bod_name}")
print(f"Лучшая для ХПК: {best_cod_name}")

Лучшая для БПК5: Naive
Лучшая для ХПК: Naive


### Permutation Importance

In [73]:
rf_bod_model = results_bod['RF'].best_estimator_
cb_bod_model = results_bod['CatBoost'].best_estimator_

Random Forest

In [74]:
imp_rf = permutation_importance(
    rf_bod_model, X_dev, y_bod_dev,
    scoring='neg_mean_absolute_error',
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

In [75]:
imp_rf_df = pd.DataFrame({
    'feature': X_dev.columns,
    'importance': imp_rf.importances_mean,
    'std': imp_rf.importances_std
}).sort_values('importance', ascending=False)

In [76]:
imp_rf_df.head(10)

,feature,importance,std
62,Синус_месяца,1.995118,0.754733
57,сыворотка,1.273654,0.571673
63,Косинус_месяца,1.056983,0.184753
61,общий_объём,1.050176,0.407908
37,Аммоний_КТ5,0.731070,0.493762
72,share_сыворотка,0.721223,0.343765
69,removal_1karta,0.689690,0.507453
41,Калий_КТ5,0.686351,0.620138
28,Взвешенные_КТ1,0.649086,0.533821
29,Взвешенные_КТ2,0.590764,0.456973


CatBoost

In [77]:
imp_cb = permutation_importance(
    cb_bod_model, X_dev, y_bod_dev,
    scoring='neg_mean_absolute_error',
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

In [78]:
imp_cb_df = pd.DataFrame({
    'feature': X_dev.columns,
    'importance': imp_cb.importances_mean,
    'std': imp_cb.importances_std
}).sort_values('importance', ascending=False)

In [79]:
imp_cb_df.head(10)

,feature,importance,std
62,Синус_месяца,2.490700,0.839234
72,share_сыворотка,2.465203,0.848817
61,общий_объём,2.409248,0.724453
57,сыворотка,1.881756,0.721798
73,cum_load_BOD_KT5,1.063386,0.211849
56,моющее_средство,0.822020,0.308583
58,творожный_цех,0.614946,0.126131
31,Взвешенные_КТ4,0.570227,0.276540
70,BOD_COD_ratio_KT1,0.502785,0.172068
21,ХПК_КТ3,0.474663,0.073897


### Сохранение признаков

In [80]:
X = final[feature_cols].copy()
module1_features = feature_cols.copy() # <-- СОХРАНЯЕМ СПИСОК ИЗ 74 ПРИЗНАКОВ

y_bod = final['target_BOD'].copy()
y_cod = final['target_COD'].copy()

# Модуль 1.5. Мультиметрический прогноз КТ5

## Список всех целевых химических показателей на КТ5

In [81]:
TARGET_COMPONENTS = {
    'BOD': 'БПК5_КТ5',
    'COD': 'ХПК_КТ5',
    'Ammonium': 'Аммоний_КТ5',
    'Phosphates': 'Фосфаты_КТ5',
    'Nitrates': 'Нитраты_КТ5',
    'Nitrites': 'Нитриты_КТ5',
    'Fats': 'Жиры_КТ5',
    'Sulfates': 'Сульфаты_КТ5'
}

## Целевые переменные со сдвигом (target) на месяц вперед (t+1)

In [82]:
for key, col_name in TARGET_COMPONENTS.items():
    if col_name in df.columns:
        df[f'target_{key}'] = df[col_name].shift(-1)
    else:
        pure_name = col_name.split('_')[0]
        if f'{pure_name}_КТ5' in panel.columns:
            df[f'target_{key}'] = df[f'{pure_name}_КТ5'].shift(-1)
        else:
            print(f"⚠ Внимание: Базовый признак {col_name} отсутствует в панельных данных!")

In [83]:
ALL_TARGET_COLS = [f'target_{k}' for k in TARGET_COMPONENTS.keys()]
BASE_KT5_COLS = list(TARGET_COMPONENTS.values())

## Фильтрация признаков и очистка от пропусков

In [84]:
EXCLUDE_FEATURES = ALL_TARGET_COLS + BASE_KT5_COLS
potential_features = [c for c in df.columns if c not in EXCLUDE_FEATURES and not c.startswith('target_')]

In [85]:
active_rows = df.dropna(subset=['БПК5_КТ5'])
nan_share = active_rows[potential_features].isna().mean()

In [86]:
feature_cols = nan_share[nan_share < 0.3].index.tolist()
print(f"Теперь пул содержит {len(feature_cols)} признаков (КТ1-КТ4 вернулись)!")

Теперь пул содержит 104 признаков (КТ1-КТ4 вернулись)!


## Цикл обучения индивидуальных ML-моделей под каждый компонент

In [87]:
module1_models = {}
module1_metrics_report = []

In [88]:
cb_params = {
    'iterations': [100, 200],
    'learning_rate': [0.02, 0.05],
    'depth': [2, 3, 4],
    'l2_leaf_reg': [5, 10, 20],
    'loss_function': ['MAE']
}
tscv = TimeSeriesSplit(n_splits=3, gap=1)

In [89]:
for key, target_col in TARGET_COMPONENTS.items():
    target_lead_col = f'target_{key}'

    if target_lead_col not in df.columns:
        continue

    # Формируем изолированную выборку под конкретный хим. компонент
    sub_df = df.dropna(subset=[target_lead_col]).copy()

    if len(sub_df) < 12:
        print(f"{target_col}: Критически мало данных для обучения ({len(sub_df)} строк). Пропуск.")
        continue

    X_target = sub_df[feature_cols].copy()
    y_target = sub_df[target_lead_col].copy()

    # Заполнение пропусков во входных фичах (ffill + медиана)
    X_target = X_target.ffill().bfill().fillna(X_target.median(numeric_only=True))
    X_target = X_target.fillna(0) # Финальный заслон

    test_size = max(5, int(np.ceil(len(X_target) * 0.20)))
    split_idx = len(X_target) - test_size

    X_train, X_test = X_target.iloc[:split_idx], X_target.iloc[split_idx:]
    y_train, y_test = y_target.iloc[:split_idx], y_target.iloc[split_idx:]

    naive_pred = sub_df[target_col].iloc[split_idx:].values

    cb = CatBoostRegressor(random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False)
    search = RandomizedSearchCV(
        estimator=cb, param_distributions=cb_params,
        n_iter=6, cv=tscv, scoring='neg_mean_absolute_error',
        random_state=RANDOM_STATE, n_jobs=-1, refit=True
    )

    try:
        search.fit(X_train, y_train)
        best_model = search.best_estimator_

        preds = best_model.predict(X_test)

        m_model = metrics(y_test, preds)
        m_naive = metrics(y_test, naive_pred)

        module1_models[key] = best_model

        module1_metrics_report.append({
            'Компонент': target_col,
            'Train_Размер': len(X_train),
            'Test_Размер': len(X_test),
            'MAE_CatBoost': round(m_model['MAE'], 3),
            'MAE_Naive': round(m_naive['MAE'], 3),
            'R2_CatBoost': round(m_model['R2'], 3)
        })

    except Exception as e:
        print(f"Сбой при обучении компонента {target_col}: {e}")

In [90]:
m1_report_df = pd.DataFrame(module1_metrics_report)
m1_report_df.to_string(index=False)

# Экспорт отчета
if REPORTS.exists():
    m1_report_df.to_csv(REPORTS / 'module1_multitarget_metrics.csv', index=False)

In [91]:
best_bod = module1_models.get('BOD', None)
best_cod = module1_models.get('COD', None)

# Модуль 2. Подбор дозировки

In [92]:
NORM_BOD = 3.0
NORM_COD = 30.0

In [93]:
best_bod = results_bod['RF'].best_estimator_
best_cod = results_cod['Ridge'].best_estimator_

In [94]:
def estimate_dosing_effect():
    try:
        dd = pd.read_csv(RAW / 'wastewater-of-dosing-devices.csv', encoding='utf-8-sig')
    except FileNotFoundError:
        return 0.4
    dd.columns = [c.strip() for c in dd.columns]
    dd = dd.rename(columns={dd.columns[0]: 'Точка'})
    pair = dd[dd['Дата'].astype(str).str.contains('24.06.2023')]
    if pair.empty: return 0.4

    before = pair[pair['Точка'].str.contains('до ввода', case=False, na=False)]
    after = pair[pair['Точка'].str.contains('после ввода', case=False, na=False)]
    if before.empty or after.empty: return 0.4

    ratio, n = 1.0, 0
    for col in before.columns:
        if col in ('Точка', 'Дата'): continue
        try:
            b = float(str(before[col].iloc[0]).replace(',', '.'))
            a = float(str(after[col].iloc[0]).replace(',', '.'))
            if b > 0 and a > 0:
                ratio *= (a / b)
                n += 1
        except (ValueError, TypeError): continue

    if n == 0: return 0.4
    mean_effect = ratio ** (1 / n)
    alpha = -np.log(max(mean_effect, 0.01)) / np.log(2)
    return float(np.clip(alpha, 0.1, 1.0))

In [95]:
ALPHA_DOSE = estimate_dosing_effect()
print(f"Калиброванный alpha для дозировки: {ALPHA_DOSE:.3f}")

Калиброванный alpha для дозировки: 0.209


In [96]:
def apply_dosing(X_row, dosing_mult, alpha=ALPHA_DOSE):
    X = X_row.copy()
    factor = 1.0 / (max(dosing_mult, 0.01) ** alpha)
    for c in X.columns:
        if any(k in c for k in ['БПК5_КТ1', 'ХПК_КТ1', 'БПК5_КТ2', 'ХПК_КТ2', 'Взвешенные', 'Аммоний']):
            X[c] = X[c] * factor
    return X

In [97]:
def predict_at_dose(dosing_mult, X_base):
    X = apply_dosing(X_base, dosing_mult)
    bod = float(best_bod.predict(X)[0])
    cod = float(best_cod.predict(X)[0])
    return bod, cod

In [98]:
X_base_row = final[module1_features].iloc[[-1]].copy() # <-- Используем module1_features вместоfeature_cols


In [99]:
def find_min_dose(X_base, target_bod=NORM_BOD, target_cod=NORM_COD):
    # Используем безопасный перебор/сканирование сетки, если brentq не находит корень
    doses_to_check = np.linspace(0.5, 20.0, 200)
    best_d = None

    for d in doses_to_check:
        b, c = predict_at_dose(d, X_base)
        if b <= target_bod and c <= target_cod:
            best_d = d
            break

    # Если на всём диапазоне норма не достигнута, возьмём максимум
    if best_d is None:
        best_d = 20.0

    return best_d, best_d, best_d

In [100]:
required_dose, d_bod, d_cod = find_min_dose(X_base_row)

## Рекомендации по дозировке

In [101]:
bod_base, cod_base = predict_at_dose(1.0, X_base_row)
print(f"При базовой дозировке (1.0×):")
print(f"БПК5 на КТ5: {bod_base:.1f} мг/дм³ ({'✅' if bod_base <= NORM_BOD else '❌'})")
print(f"ХПК на КТ5:  {cod_base:.1f} мгО/дм³ ({'✅' if cod_base <= NORM_COD else '❌'})")
print("\n")

if required_dose and required_dose < 20.0:
    bod_rec, cod_rec = predict_at_dose(required_dose, X_base_row)
    print(f"Рекомендуемая дозировка: {required_dose:.2f} ×")
    print(f"Прогноз БПК5: {bod_rec:.1f} мг/дм³")
    print(f"Прогноз ХПК:  {cod_rec:.1f} мгО/дм³")
else:
    print("⚠ Даже при максимальной дозировке нормативы не могут быть полностью достигнуты.")
    print("Рекомендуется рассмотреть дополнительную очистку.")

При базовой дозировке (1.0×):
БПК5 на КТ5: 117.2 мг/дм³ (❌)
ХПК на КТ5:  236.2 мгО/дм³ (❌)


⚠ Даже при максимальной дозировке нормативы не могут быть полностью достигнуты.
Рекомендуется рассмотреть дополнительную очистку.


In [102]:
doses = np.linspace(0.5, 10.0, 10)
results_curve = []
for d in doses:
    b, c = predict_at_dose(d, X_base_row)
    results_curve.append({'dose': d, 'BOD': b, 'COD': c})
curve_df = pd.DataFrame(results_curve)
curve_df.to_csv(REPORTS / 'module2_dose_curve.csv', index=False)

# Модуль 3. Прогноз донных отложений

In [103]:
sed_targets = [c for c in df.columns if c.startswith('target_SED_') and 'КТ5' in c]

In [104]:
if sed_targets:
    module3_models = {}
    module3_metrics = []
    base_features = [c for c in df.columns if not c.startswith('target_') and not c.startswith('SED_') and c not in ['БПК5_КТ5', 'ХПК_КТ5']]

    for target_col in sed_targets:
        target_name = target_col.replace('target_', '')
        # КРИТИЧЕСКОЕ ИСПРАВЛЕНИЕ: Сбрасываем строковый временной индекс в обычную колонку перед ffill!
        sub = df.dropna(subset=[target_col]).copy().reset_index()

        if len(sub) < 10:
            continue

        current_col = target_name
        X_cols = base_features + [current_col]
        X_cols = [c for c in X_cols if c in sub.columns]

        Xs = sub[X_cols].copy()
        ys = sub[target_col].copy()

        # Теперь ffill() отрабатывает на 100% корректно без дыр
        Xs = Xs.ffill().bfill().fillna(0)

        test_size_s = max(3, int(np.ceil(len(Xs) * 0.25)))
        split_s = len(Xs) - test_size_s

        Xs_dev, Xs_test = Xs.iloc[:split_s], Xs.iloc[split_s:]
        ys_dev, ys_test = ys.iloc[:split_s], ys.iloc[split_s:]

        if len(Xs_dev) < 5: continue

        naive_s = sub.loc[Xs_test.index, current_col].values

        pipe = Pipeline([('sc', StandardScaler()), ('m', Ridge())])
        tscv_s = TimeSeriesSplit(n_splits=min(3, len(Xs_dev) - 1), gap=1)
        search_s = RandomizedSearchCV(
            pipe, {'m__alpha': np.logspace(-3, 3, 15)},
            n_iter=10, cv=tscv_s, scoring='neg_mean_absolute_error',
            random_state=RANDOM_STATE, refit=True
        )

        try:
            search_s.fit(Xs_dev, ys_dev)
            pred_s = search_s.best_estimator_.predict(Xs_test)

            m_model = metrics(ys_test, pred_s)
            m_naive = metrics(ys_test, naive_s)

            module3_models[target_name] = {
                'model': search_s.best_estimator_,
                'features': X_cols,
                'metrics': m_model,
                'naive': m_naive,
            }
            module3_metrics.append({
                'target': target_name.replace('SED_',''),
                'n_train': len(Xs_dev), 'n_test': len(Xs_test),
                'MAE_model': m_model['MAE'], 'MAE_naive': m_naive['MAE']
            })
        except Exception as e:
            print(f"⚠ Ошибка обучения {target_name}: {e}")

    print("\n")
    if module3_metrics:
        m3_df = pd.DataFrame(module3_metrics)
        print("Результаты валидации моделей донных отложений:")
        print(m3_df.head(8).to_string(index=False))
        if MODELS.exists():
            joblib.dump(module3_models, MODELS / 'module3_sediments.pkl')
    else:
        print("Не удалось обучить модели Модуля 3.")


⚠ Ошибка обучения SED_Натрий_мг/кг_КТ5: Input contains NaN.


Результаты валидации моделей донных отложений:
                                   target  n_train  n_test   MAE_model  MAE_naive
             Массовая_доля_серы_мг/кг_КТ5       23       8  195.665118     0.7125
       Водородный_показатель_рН_ед.рН_КТ5       23       8    0.299576     0.0250
         Массовая_доля_нитратов_мг/кг_КТ5       23       8   95.626927     0.2375
                Массовая_доля_влаги_%_КТ5       23       8    7.303571     1.4000
Массовая_доля_аммонийного_азота_мг/кг_КТ5       23       8   27.645468     2.1000
                      Сухой_остаток_%_КТ5       23       8    7.499747     1.4000
              Органическое_вещество_%_КТ5       23       8   35.522884     0.6250
               Фосфор_подвижный_мг/кг_КТ5       23       8 1164.328450     8.7125


# Построение сценариев

In [105]:
def scenario_predict(base_X, dosing_mult=1.0, volume_mult=1.0, temp_delta=0.0, precip_mult=1.0):
    X = base_X.copy()
    X = apply_dosing(X, dosing_mult)
    if 'общий_объём' in X.columns:
        X['общий_объём'] = X['общий_объём'] * volume_mult
    if 'T_mean' in X.columns:
        X['T_mean'] = X['T_mean'] + temp_delta
    if 'precip_sum' in X.columns:
        X['precip_sum'] = X['precip_sum'] * precip_mult
    return float(best_bod.predict(X)[0]), float(best_cod.predict(X)[0])

In [106]:
scenarios = {
    'Baseline': {'dosing_mult': 1.0, 'volume_mult': 1.0, 'temp_delta': 0.0, 'precip_mult': 1.0},
    'Дозировка ×2': {'dosing_mult': 2.0, 'volume_mult': 1.0, 'temp_delta': 0.0, 'precip_mult': 1.0},
    'Дозировка ×3': {'dosing_mult': 3.0, 'volume_mult': 1.0, 'temp_delta': 0.0, 'precip_mult': 1.0},
    'Объём +20%': {'dosing_mult': 1.0, 'volume_mult': 1.2, 'temp_delta': 0.0, 'precip_mult': 1.0},
    'Тепло +2°C': {'dosing_mult': 1.0, 'volume_mult': 1.0, 'temp_delta': 2.0, 'precip_mult': 1.0},
    'Холод −2°C': {'dosing_mult': 1.0, 'volume_mult': 1.0, 'temp_delta': -2.0, 'precip_mult': 1.0},
    'Осадки +20%': {'dosing_mult': 1.0, 'volume_mult': 1.0, 'temp_delta': 0.0, 'precip_mult': 1.2},
}

In [107]:
scen_results = []

for name, params in scenarios.items():
  b, c = scenario_predict(X_base_row, **params)
  scen_results.append({
      'Scenario': name,
      'BOD_KT5': round(b, 2),
      'COD_KT5': round(c, 2),
      'BOD_norm': b <= NORM_BOD,
      'COD_norm': c <= NORM_COD,
})

In [108]:
scen_df = pd.DataFrame(scen_results)
print("Прогнозные сценарии очистки на КТ5:")
scen_df

Прогнозные сценарии очистки на КТ5:


,Scenario,BOD_KT5,COD_KT5,BOD_norm,COD_norm
0,Baseline,117.18,236.24,False,False
1,Дозировка ×2,117.56,240.39,False,False
2,Дозировка ×3,118.61,242.56,False,False
3,Объём +20%,123.72,239.82,False,False
4,Тепло +2°C,117.18,236.24,False,False
5,Холод −2°C,117.18,236.24,False,False
6,Осадки +20%,117.18,236.24,False,False


# Итоговый отчёт по результатам трёх модулей

In [109]:
print(f"Параметры панели: {panel.shape}")
print(f"Модуль 1 (Прогноз) — Обучающая выборка: {len(X_dev)} строк, Тестовая: {len(X_test)}")
print(f"Модуль 2 (Дозировка) — Эффективный коэффициент alpha равен: {ALPHA_DOSE:.3f}")
if module3_metrics:
  print(f"Модуль 3 (Отложения) — Успешно обучено моделей: {len(module3_models)}")

Параметры панели: (348, 315)
Модуль 1 (Прогноз) — Обучающая выборка: 21 строк, Тестовая: 6
Модуль 2 (Дозировка) — Эффективный коэффициент alpha равен: 0.209
Модуль 3 (Отложения) — Успешно обучено моделей: 14


# Тестирование моделей

In [110]:
!pip install -q gradio plotly reportlab

In [111]:
import gradio as gr
import numpy as np
import pandas as pd

def web_inference_interface(volume, whey, curd_shop, detergent, temp, precip, dose_mult, vol_mult):
    """
    Принимает значения из веб-форм, формирует вектор признаков,
    применяет логику дозирования и возвращает результаты для отображения.
    """
    base_data = {}
    for col in feature_cols:
        base_data[col] = [0.0] # заполняем нулями по умолчанию

    df_input = pd.DataFrame(base_data)

    if 'общий_объём' in df_input.columns: df_input['общий_объём'] = float(volume) * vol_mult
    if 'сыворотка' in df_input.columns: df_input['сыворотка'] = float(whey)
    if 'творожный_цех' in df_input.columns: df_input['творожный_цех'] = float(curd_shop)
    if 'моющее_средство' in df_input.columns: df_input['моющее_средство'] = float(detergent)
    if 'T_mean' in df_input.columns: df_input['T_mean'] = float(temp)
    if 'precip_sum' in df_input.columns: df_input['precip_sum'] = float(precip)
    if 'share_сыворотка' in df_input.columns: df_input['share_сыворотка'] = float(whey) / max(float(volume), 1.0)

    if 'Синус_месяца' in df_input.columns: df_input['Синус_месяца'] = np.sin(2 * np.pi * 5 / 12)
    if 'Косинус_месяца' in df_input.columns: df_input['Косинус_месяца'] = np.cos(2 * np.pi * 5 / 12)

    df_engineered = apply_dosing(df_input, dose_mult, alpha=ALPHA_DOSE)

    report_output = "📊 ПРОГНОЗ ПОКАЗАТЕЛЕЙ НА КТ5:\n" + "-"*40 + "\n"

    ru_names = {
        'BOD': 'БПК5 (Норма ≤ 3.0 мг/дм³)', 'COD': 'ХПК (Норма ≤ 30.0 мгО/дм³)',
        'Ammonium': 'Аммоний', 'Phosphates': 'Фосфаты', 'Nitrates': 'Нитраты',
        'Nitrites': 'Нитриты', 'Fats': 'Жиры', 'Sulfates': 'Сульфаты'
    }

    for key, model in module1_models.items():
        df_aligned = df_engineered[model.feature_names_]
        pred_val = float(model.predict(df_aligned)[0])

        status = "[\/] Норма"
        if key == 'BOD' and pred_val > 3.0: status = "[X] Превышение"
        if key == 'COD' and pred_val > 30.0: status = "[X] Превышение"

        report_output += f"{ru_names.get(key, key)}: {pred_val:.2f} — {status}\n"

    report_output += "ПРОГНОЗ НАКОПЛЕНИЯ В ДОННЫХ ОТЛОЖЕНИЯХ (КТ5):\n" + "-"*40 + "\n"

    if module3_models:
        for target_name, info in module3_models.items():
            model_m3 = info['model']
            f_list = info['features']

            df_m3 = pd.DataFrame(index=[0])
            for col in f_list:
                df_m3[col] = df_input[col].values if col in df_input.columns else 0.0

            pred_val_m3 = float(model_m3.predict(df_m3)[0])
            clean_name = target_name.replace('SED_', '').replace('_КТ5', '')
            report_output += f"{clean_name}: {pred_val_m3:.2f}\n"
    else:
        report_output += "⚠ Модели отложений не задействованы (используйте исторический стабильный контур Naive)"

    return report_output

# 2. Проектирование визуального интерфейса с помощью блоков Gradio
with gr.Blocks(theme=gr.themes.Soft(), title="Golden Meadows AI") as demo:
    gr.Markdown("# 🌾 Аналитическая система эко-мониторинга «Golden Meadows»")
    gr.Markdown("Интегрированный инференс-модуль предиктивного анализа стоков и оптимизации хим. реагентов.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🏭 Параметры производства и погоды")
            inp_vol = gr.Number(label="Базовый объем стоков (м³)", value=45000)
            inp_whey = gr.Number(label="Сброс сыворотки (м³)", value=1100)
            inp_curd = gr.Number(label="Поток творожного цеха (м³)", value=10500)
            inp_det = gr.Number(label="Моющие средства (м³)", value=38)
            inp_temp = gr.Slider(label="Температура воздуха (°C)", minimum=-25, maximum=30, value=12)
            inp_precip = gr.Number(label="Атмосферные осадки (мм)", value=42)

        with gr.Column():
            gr.Markdown("### ⚙ Сценарии What-If и Управление")
            inp_dose = gr.Slider(label="Кратность дозирования реагента (Модуль 2)", minimum=0.1, maximum=10.0, value=1.0, step=0.1)
            inp_vol_mult = gr.Slider(label="Масштабирование притока (коэффициент объема)", minimum=0.5, maximum=2.0, value=1.0, step=0.1)

            btn_run = gr.Button("Сгенерировать комплексный отчет", variant="primary")

    with gr.Row():
        txt_output = gr.Textbox(label="Аналитическое заключение системы искусственного интеллекта", lines=18, max_lines=30)

    btn_run.click(
        fn=web_inference_interface,
        inputs=[inp_vol, inp_whey, inp_curd, inp_det, inp_temp, inp_precip, inp_dose, inp_vol_mult],
        outputs=txt_output
    )

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a820b9ed66b0d6f532.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Экспорт моделей

In [112]:
if 'module1_models' in locals() and module1_models:
    # Сохраняем каждую модель под своим именем в отдельные файлы
    for component_key, model_instance in module1_models.items():
        model_filename = MODELS / f"module1_model_{component_key}.pkl"
        joblib.dump(model_instance, model_filename)
    print(f"Модуль 1: Успешно сохранено {len(module1_models)} моделей CatBoost в файлы module1_model_*.pkl")
else:
    print("Ошибка: Словарь module1_models не найден в памяти или пуст.")

Модуль 1: Успешно сохранено 8 моделей CatBoost в файлы module1_model_*.pkl


In [113]:
if 'ALPHA_DOSE' in locals():
    dosing_config = {
        'ALPHA_DOSE': ALPHA_DOSE,
        'NORM_BOD': 3.0,
        'NORM_COD': 30.0
    }
    joblib.dump(dosing_config, MODELS / 'module2_dosing_config.pkl')
    print(f"Модуль 2: Конфигурация дозирования (alpha = {ALPHA_DOSE:.3f}) сохранена в module2_dosing_config.pkl")
else:
    print("Ошибка: Переменная ALPHA_DOSE не найдена в памяти.")

Модуль 2: Конфигурация дозирования (alpha = 0.209) сохранена в module2_dosing_config.pkl


In [114]:
if 'module3_models' in locals() and module3_models:
    # Сохраняем весь словарь моделей Ridge вместе с их метаданными (списками фич)
    joblib.dump(module3_models, MODELS / 'module3_sediments.pkl')
    print(f"Модуль 3: Успешно сохранено {len(module3_models)} моделей Ridge в файл module3_sediments.pkl")
else:
    print("⚠ Модуль 3 не был сохранен (недостаточно данных или пустой словарь).")

print("Все доступные артефакты успешно записаны в директорию:")
print(f"{MODELS.resolve()}")

Модуль 3: Успешно сохранено 14 моделей Ridge в файл module3_sediments.pkl
Все доступные артефакты успешно записаны в директорию:
/content/drive/MyDrive/Projects/golden-meadows/models
